In [13]:
import pandas as pd
import numpy as np

train = pd.read_csv('./ML/data/bike_train.csv')
train.info()
train.head()

<class 'pandas.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   datetime    10886 non-null  str    
 1   season      10886 non-null  int64  
 2   holiday     10886 non-null  int64  
 3   workingday  10886 non-null  int64  
 4   weather     10886 non-null  int64  
 5   temp        10886 non-null  float64
 6   atemp       10886 non-null  float64
 7   humidity    10886 non-null  int64  
 8   windspeed   10886 non-null  float64
 9   casual      10886 non-null  int64  
 10  registered  10886 non-null  int64  
 11  count       10886 non-null  int64  
dtypes: float64(3), int64(8), str(1)
memory usage: 1020.7 KB


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [14]:
train['datetime'] = pd.to_datetime(train['datetime'])

train['year'] = train.datetime.apply(lambda x: x.year)
train['month'] = train.datetime.apply(lambda x: x.month)
train['day'] = train.datetime.apply(lambda x: x.day)
train['hour'] = train.datetime.apply(lambda x: x.hour)

train.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,year,month,day,hour
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16,2011,1,1,0
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40,2011,1,1,1
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32,2011,1,1,2
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13,2011,1,1,3
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1,2011,1,1,4


In [15]:
train_new = train.drop(['datetime', 'casual', 'registered'], axis=1)

In [16]:
def rmsle(y, pred):
    log_y = np.log1p(y)
    log_pred = np.log1p(pred)
    squared_error = (log_y - log_pred) ** 2
    rmsle = np.sqrt(np.mean(squared_error))
    return rmsle

def rmse(y, pred):
    return np.sqrt(mean_squared_error(y, pred))

def evaluate_regr(y, pred):
    rmsle_val = rmsle(y, pred)
    rmse_val = rmse(y, pred)
    mae_val = mean_absolute_error(y, pred)
    print('RMSLE:', rmsle_val, 'RMSE:', rmse_val, 'MAE:', mae_val)

In [17]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

y_target = train_new['count']
X_features = train_new.drop(['count'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X_features, y_target,
                                                     test_size=0.3, random_state=42)

lr_reg = LinearRegression()
lr_reg.fit(X_train, y_train)
pred = lr_reg.predict(X_test)

evaluate_regr(y_test, pred)

RMSLE: 1.173590411228646 RMSE: 141.26089979006076 MAE: 105.67593480324634


C:\Users\KDS23\AppData\Local\Temp\ipykernel_14876\2350828039.py:3: RuntimeWarning: invalid value encountered in log1p
  log_pred = np.log1p(pred)


In [19]:
pred = lr_reg.predict(X_test)
pred = np.where(pred < 0, 0, pred)

evaluate_regr(y_test, pred)

RMSLE: 1.2930080007955158 RMSE: 140.59763832962352 MAE: 103.99748292136346


In [20]:
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(n_estimators=500, random_state=42)
rf_reg.fit(X_train, y_train)
pred_rf = rf_reg.predict(X_test)

evaluate_regr(y_test, pred_rf)

RMSLE: 0.3485232227389425 RMSE: 42.45632596326296 MAE: 27.31931108389467


In [21]:
y_target_log = np.log1p(y_target)

X_train, X_test, y_train_log, y_test_log = train_test_split(X_features, y_target_log,
                                                             test_size=0.3, random_state=42)

lr_reg = LinearRegression()
lr_reg.fit(X_train, y_train_log)
pred_log = lr_reg.predict(X_test)

pred = np.expm1(pred_log)
y_test_exp = np.expm1(y_test_log)
pred = np.where(pred < 0, 0, pred)

evaluate_regr(y_test_exp, pred)

RMSLE: 1.0202175464634047 RMSE: 160.72955754312727 MAE: 108.2356846915191
